# poster figures

In [ ]:
import sys
sys.path.insert(0, r"C:\Users\RBO\repos\LBM-Suite2p-Python")

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb
from scipy.ndimage import distance_transform_edt

from lbm_suite2p_python.zplane import (
    normalize99,
    plot_masks,
    plot_projection,
    plot_rastermap,
    stat_to_mask,
    mask_overlay,
    feather_mask,
    random_colors_for_mask,
    get_background_image,
)
from lbm_suite2p_python.utils import bin1d

plt.style.use("dark_background")

RESULTS = Path(r"D:\demo\results")
SAVE = RESULTS / "poster"
SAVE.mkdir(exist_ok=True)
NUM_PLANES = 14
DPI = 300


def load_plane(idx):
    d = RESULTS / f"plane{idx:02d}_stitched"
    ops = np.load(d / "ops.npy", allow_pickle=True).item()
    stat = np.load(d / "stat.npy", allow_pickle=True)
    iscell = np.load(d / "iscell.npy")
    return dict(
        ops=ops, stat=stat, iscell=iscell,
        F=np.load(d / "F.npy"),
        dff=np.load(d / "dff.npy"),
        spks=np.load(d / "spks.npy"),
    )


def save_img(fig, name):
    p = SAVE / f"{name}.png"
    fig.savefig(p, dpi=DPI, facecolor=fig.get_facecolor(), bbox_inches="tight", pad_inches=0)
    print(f"saved: {p}")
    plt.show()

## individual projection images per plane
bare images â€” no axes, no borders, just the data on black

In [ ]:
# save bare projection images for every plane
for pidx in range(1, NUM_PLANES + 1):
    p = load_plane(pidx)
    ops = p["ops"]

    for key, label in [("max_proj", "maxproj"), ("Vcorr", "correlation"), ("meanImg", "mean"), ("meanImgE", "mean_enhanced")]:
        img = ops.get(key)
        if img is None:
            continue
        fig, ax = plt.subplots(figsize=(6, 6), facecolor="black")
        ax.set_facecolor("black")
        cmap = "inferno" if key == "Vcorr" else "gray"
        ax.imshow(normalize99(img), cmap=cmap, interpolation="nearest")
        ax.axis("off")
        for spine in ax.spines.values():
            spine.set_visible(False)
        fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
        save_img(fig, f"plane{pidx:02d}_{label}")
        plt.close(fig)

## volume projections â€” full stack collapsed
max/mean intensity projection and correlation across all z-planes

In [ ]:
# embed each plane's cropped images into full FOV, then collapse across z
Ly_full, Lx_full = 550, 448

def stack_across_planes(img_key):
    """collect one image type from all planes, embedded in full FOV coords."""
    stack = []
    for pidx in range(1, NUM_PLANES + 1):
        ops = load_plane(pidx)["ops"]
        img = ops.get(img_key)
        if img is None:
            continue
        yrange = ops.get("yrange", [0, Ly_full])
        xrange = ops.get("xrange", [0, Lx_full])
        cropped_keys = {"max_proj", "Vcorr"}
        if img_key in cropped_keys:
            # embed cropped image into full FOV
            canvas = np.zeros((Ly_full, Lx_full), dtype=np.float32)
            y0, y1 = int(yrange[0]), int(yrange[0]) + img.shape[0]
            x0, x1 = int(xrange[0]), int(xrange[0]) + img.shape[1]
            canvas[y0:y1, x0:x1] = img
        else:
            canvas = img.astype(np.float32)
        stack.append(canvas)
    return np.array(stack)


# -- max projection across z-stack --
maxproj_stack = stack_across_planes("max_proj")
vol_maxproj = np.max(maxproj_stack, axis=0)

# -- mean of mean images --
mean_stack = stack_across_planes("meanImg")
vol_mean = np.mean(mean_stack, axis=0)

# -- max correlation across z --
vcorr_stack = stack_across_planes("Vcorr")
vol_corr = np.max(vcorr_stack, axis=0)

# -- mean of enhanced mean --
meane_stack = stack_across_planes("meanImgE")
vol_meane = np.mean(meane_stack, axis=0)

volume_imgs = [
    ("vol_maxproj", vol_maxproj, "gray"),
    ("vol_mean", vol_mean, "gray"),
    ("vol_mean_enhanced", vol_meane, "gray"),
    ("vol_correlation", vol_corr, "inferno"),
]

for name, img, cmap in volume_imgs:
    fig, ax = plt.subplots(figsize=(6, 6), facecolor="black")
    ax.set_facecolor("black")
    ax.imshow(normalize99(img), cmap=cmap, interpolation="nearest")
    ax.axis("off")
    for spine in ax.spines.values():
        spine.set_visible(False)
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    save_img(fig, name)
    plt.close(fig)

## mask overlays per plane
uses `plot_masks` from lbm_suite2p_python â€” HSV colored ROIs blended on mean image

In [ ]:
for pidx in range(1, NUM_PLANES + 1):
    p = load_plane(pidx)
    ops, stat, iscell = p["ops"], p["stat"], p["iscell"]
    mask_idx = iscell[:, 0].astype(bool)
    n_acc = int(mask_idx.sum())

    # use mean image as background
    bg = ops["meanImg"]
    plot_masks(
        bg, stat, mask_idx,
        savepath=SAVE / f"plane{pidx:02d}_masks.png",
        title=f"Plane {pidx} â€” {n_acc} ROIs",
    )
    print(f"plane {pidx}: {n_acc} accepted")

## masks on max projection

In [ ]:
for pidx in range(1, NUM_PLANES + 1):
    p = load_plane(pidx)
    ops, stat, iscell = p["ops"], p["stat"], p["iscell"]
    mask_idx = iscell[:, 0].astype(bool)
    n_acc = int(mask_idx.sum())

    bg, yoff, xoff = get_background_image(ops, "max_proj")

    # shift stat coords into cropped space
    stat_shifted = []
    for s in stat:
        sc = dict(s)
        sc["ypix"] = s["ypix"] - yoff
        sc["xpix"] = s["xpix"] - xoff
        sc["lam"] = s["lam"]
        stat_shifted.append(sc)

    plot_masks(
        bg, stat_shifted, mask_idx,
        savepath=SAVE / f"plane{pidx:02d}_masks_maxproj.png",
        title=f"Plane {pidx} â€” {n_acc} ROIs (max proj)",
    )
    print(f"plane {pidx}: {n_acc} accepted")

## high-res copies of existing summary PNGs
load the already-generated summary images, re-render at poster DPI on black

In [ ]:
summary_files = [
    "all_planes_masks.png",
    "orthoslices.png",
    "rastermap.png",
    "roi_map_3d.png",
    "volume_quality_diagnostics.png",
    "volume_quality_metrics.png",
    "volume_trace_analysis.png",
]

for fname in summary_files:
    src = RESULTS / fname
    if not src.exists():
        print(f"skip: {fname}")
        continue
    img = plt.imread(str(src))
    h, w = img.shape[:2]
    # scale figure so saved image is at least 300 DPI equivalent
    fig_w = max(w / DPI, 10)
    fig_h = max(h / DPI, 6)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), facecolor="black")
    ax.set_facecolor("black")
    ax.imshow(img, interpolation="lanczos")
    ax.axis("off")
    for spine in ax.spines.values():
        spine.set_visible(False)
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    save_img(fig, f"summary_{Path(fname).stem}")
    plt.close(fig)

## per-plane summary images (high-res copies)

In [ ]:
# collect all per-plane PNGs worth printing
per_plane_images = [
    "01_correlation.png",
    "01_correlation_segmentation.png",
    "02_max_projection.png",
    "02_max_projection_segmentation.png",
    "03_mean.png",
    "03_mean_segmentation.png",
    "04_mean_enhanced.png",
    "04_mean_enhanced_segmentation.png",
    "05_quality_diagnostics.png",
]

# do a few representative planes
selected_planes = [1, 3, 5, 8, 12]

for pidx in selected_planes:
    plane_dir = RESULTS / f"plane{pidx:02d}_stitched"
    for fname in per_plane_images:
        src = plane_dir / fname
        if not src.exists():
            continue
        img = plt.imread(str(src))
        h, w = img.shape[:2]
        fig_w = max(w / DPI, 8)
        fig_h = max(h / DPI, 6)
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), facecolor="black")
        ax.set_facecolor("black")
        ax.imshow(img, interpolation="lanczos")
        ax.axis("off")
        for spine in ax.spines.values():
            spine.set_visible(False)
        fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
        stem = Path(fname).stem
        save_img(fig, f"plane{pidx:02d}_{stem}")
        plt.close(fig)

## activity heatmap â€” all planes, dark style

In [ ]:
# single-plane heatmap with dark styling
PLANE_IDX = 3

p = load_plane(PLANE_IDX)
accepted = np.where(p["iscell"][:, 0] > 0.5)[0]
dff_acc = p["dff"][accepted]
fs = p["ops"].get("fs", 30.0)

# z-score
mu = dff_acc.mean(axis=1, keepdims=True)
sd = dff_acc.std(axis=1, keepdims=True) + 1e-6
dff_z = (dff_acc - mu) / sd

# sort by peak time
order = np.argsort(np.argmax(dff_z, axis=1))
dff_sorted = dff_z[order]

vmax = np.percentile(dff_sorted, 99)
vmin = np.percentile(dff_sorted, 1)
time = np.arange(dff_sorted.shape[1]) / fs

fig, ax = plt.subplots(figsize=(14, 6), facecolor="black")
ax.set_facecolor("black")
im = ax.imshow(
    dff_sorted, aspect="auto", cmap="inferno",
    vmin=vmin, vmax=vmax,
    extent=[0, time[-1], len(accepted), 0],
    interpolation="none", rasterized=True,
)
ax.set_xlabel("Time (s)", color="white", fontsize=10)
ax.set_ylabel("Neuron", color="white", fontsize=10)
ax.tick_params(colors="white")
for spine in ax.spines.values():
    spine.set_color("white")

cbar = plt.colorbar(im, ax=ax, shrink=0.5, pad=0.02)
cbar.set_label("z-scored dF/F", color="white", fontsize=9)
cbar.ax.tick_params(colors="white")
cbar.outline.set_edgecolor("white")

ax.set_title(
    f"Plane {PLANE_IDX} â€” {len(accepted)} neurons",
    color="white", fontsize=12, fontweight="bold",
)
plt.tight_layout()
save_img(fig, f"heatmap_plane{PLANE_IDX:02d}")

## volume heatmap â€” all planes stacked

In [ ]:
all_dff = []
boundaries = [0]

for pidx in range(1, NUM_PLANES + 1):
    p = load_plane(pidx)
    acc = np.where(p["iscell"][:, 0] > 0.5)[0]
    d = p["dff"][acc]
    mu = d.mean(axis=1, keepdims=True)
    sd = d.std(axis=1, keepdims=True) + 1e-6
    all_dff.append((d - mu) / sd)
    boundaries.append(boundaries[-1] + len(acc))

all_dff = np.vstack(all_dff)
fs = load_plane(1)["ops"].get("fs", 30.0)
time = np.arange(all_dff.shape[1]) / fs

vmin = np.percentile(all_dff, 1)
vmax = np.percentile(all_dff, 99)

fig, ax = plt.subplots(figsize=(18, 10), facecolor="black")
ax.set_facecolor("black")

im = ax.imshow(
    all_dff, aspect="auto", cmap="inferno",
    vmin=vmin, vmax=vmax,
    extent=[0, time[-1], all_dff.shape[0], 0],
    interpolation="none", rasterized=True,
)

# plane boundary lines and labels
for i, b in enumerate(boundaries[1:-1]):
    ax.axhline(b, color="white", linewidth=0.4, alpha=0.5)
    mid = (boundaries[i] + b) / 2
    ax.text(-time[-1] * 0.015, mid, f"P{i+1}", color="white",
            fontsize=6, va="center", ha="right", fontweight="bold")

ax.set_xlabel("Time (s)", color="white", fontsize=11, fontweight="bold")
ax.set_ylabel("Neuron", color="white", fontsize=11, fontweight="bold")
ax.tick_params(colors="white")
for spine in ax.spines.values():
    spine.set_color("white")

cbar = plt.colorbar(im, ax=ax, shrink=0.4, pad=0.015)
cbar.set_label("z-scored dF/F", color="white", fontsize=9)
cbar.ax.tick_params(colors="white")
cbar.outline.set_edgecolor("white")

ax.set_title(
    f"{all_dff.shape[0]} neurons across {NUM_PLANES} planes",
    color="white", fontsize=13, fontweight="bold",
)
plt.tight_layout()
save_img(fig, "heatmap_volume")

## rastermap (if model exists)

In [ ]:
model_path = RESULTS / "rastermap_model.npy"
if model_path.exists():
    # load a representative plane's spks for rastermap
    # or use the volume-level model if it was fit on all planes
    model = np.load(model_path, allow_pickle=True).item()

    # stack all accepted spks
    all_spks = []
    for pidx in range(1, NUM_PLANES + 1):
        p = load_plane(pidx)
        acc = np.where(p["iscell"][:, 0] > 0.5)[0]
        all_spks.append(p["spks"][acc])
    all_spks = np.vstack(all_spks)

    fs = load_plane(1)["ops"].get("fs", 30.0)

    plot_rastermap(
        all_spks, model, fps=fs,
        vmin=0, vmax=0.8,
        save_path=SAVE / "rastermap_hires.png",
        title="Rastermap â€” All Planes",
    )
    print(f"saved rastermap ({all_spks.shape[0]} neurons)")
else:
    print("no rastermap model found")

## dF/F traces
per-cell z-scored then clipped to [0, max] so every neuron's activity is visible at the same scale.
50, 75, 150 cells x 10s, 25s windows.

In [ ]:
from lbm_suite2p_python.zplane import plot_traces
from scipy.ndimage import uniform_filter1d

PLANE_IDX = 3

p = load_plane(PLANE_IDX)
iscell = p["iscell"]
F_raw = p["F"]
Fneu = np.load(RESULTS / f"plane{PLANE_IDX:02d}_stitched" / "Fneu.npy")
accepted = np.where(iscell[:, 0] > 0.5)[0]
fs = p["ops"].get("fs", 30.0)

# neuropil correction
F_corr = F_raw - 0.7 * Fneu

# per-cell: subtract rolling 8th percentile baseline, divide, z-score, clip
from scipy.ndimage import percentile_filter
bwin = int(60 * fs) | 1  # odd window

dff_clean = np.zeros_like(F_corr)
for i in range(F_corr.shape[0]):
    baseline = percentile_filter(F_corr[i], percentile=8, size=bwin)
    baseline = np.maximum(baseline, 1.0)
    raw_dff = (F_corr[i] - baseline) / baseline

    # z-score per cell so all cells have comparable amplitude
    mu, sd = raw_dff.mean(), raw_dff.std()
    z = (raw_dff - mu) / (sd + 1e-6)

    # clip negatives, light smooth to reduce noise
    z = np.clip(z, 0, None)
    z = uniform_filter1d(z, size=max(int(0.3 * fs), 1))  # ~300ms smooth
    dff_clean[i] = z

# rank by peak z-score (most visually obvious transients)
peaks = np.max(dff_clean[accepted], axis=1)
top = accepted[np.argsort(peaks)[::-1]]

cell_counts = [50, 75, 150]
windows = [10, 25]

for nc in cell_counts:
    indices = top[:nc]
    for win_s in windows:
        name = f"traces_p{PLANE_IDX:02d}_{nc}cells_{win_s}s"
        plot_traces(
            dff_clean,
            save_path=SAVE / f"{name}.png",
            cell_indices=indices,
            fps=fs,
            window=win_s,
            title=f"Plane {PLANE_IDX} â€” {nc} neurons, {win_s}s",
            scale_bar_unit="z-score",
        )
        print(f"saved {name}")

print("done")

## masks + traces composite panels
left: mask overlay on mean image, right: z-scored dF/F traces for the same cells.
one figure per plane, saved to `poster/mask_traces/`.

In [ ]:
from scipy.ndimage import percentile_filter, uniform_filter1d
from lbm_suite2p_python.zplane import get_color_permutation

OUTDIR = SAVE / "mask_traces"
OUTDIR.mkdir(exist_ok=True)

N_CELLS = 75
WINDOW_S = 25  # seconds of traces to show

selected_planes = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]

for pidx in selected_planes:
    p = load_plane(pidx)
    ops, stat, iscell = p["ops"], p["stat"], p["iscell"]
    F_raw, Fneu = p["F"], np.load(RESULTS / f"plane{pidx:02d}_stitched" / "Fneu.npy")
    Ly, Lx = ops["Ly"], ops["Lx"]
    fs = ops.get("fs", 30.0)
    accepted = np.where(iscell[:, 0] > 0.5)[0]
    n_show = min(N_CELLS, len(accepted))

    # -- compute clean dF/F --
    F_corr = F_raw - 0.7 * Fneu
    bwin = int(60 * fs) | 1
    dff_clean = np.zeros_like(F_corr)
    for i in range(F_corr.shape[0]):
        bl = percentile_filter(F_corr[i], percentile=8, size=bwin)
        bl = np.maximum(bl, 1.0)
        raw = (F_corr[i] - bl) / bl
        mu, sd = raw.mean(), raw.std()
        z = (raw - mu) / (sd + 1e-6)
        z = np.clip(z, 0, None)
        z = uniform_filter1d(z, size=max(int(0.3 * fs), 1))
        dff_clean[i] = z

    # pick top cells by peak z-score
    peaks = np.max(dff_clean[accepted], axis=1)
    top = accepted[np.argsort(peaks)[::-1]][:n_show]

    # -- assign colors (same for mask and traces) --
    cmap_inst = plt.get_cmap("tab10")
    raw_colors = cmap_inst(np.linspace(0, 1, n_show))[:, :3]
    perm = get_color_permutation(n_show)
    colors = raw_colors[perm]

    # -- build mask canvas --
    bg = ops["meanImg"]
    vmin_bg, vmax_bg = np.nanpercentile(bg, [1, 99])
    norm_bg = np.clip((bg - vmin_bg) / (vmax_bg - vmin_bg + 1e-6), 0, 1)
    norm_bg = np.nan_to_num(norm_bg, nan=0.0)
    canvas = np.tile(norm_bg, (3, 1, 1)).transpose(1, 2, 0).copy()

    for ci, cell_idx in enumerate(top):
        s = stat[cell_idx]
        yy, xx, lam = s["ypix"], s["xpix"], s["lam"]
        valid = (yy >= 0) & (yy < Ly) & (xx >= 0) & (xx < Lx)
        yy, xx, lam = yy[valid], xx[valid], lam[valid]
        lam = lam / (lam.max() + 1e-10)
        col = colors[ci]
        for k in range(3):
            canvas[yy, xx, k] = 0.5 * canvas[yy, xx, k] + 0.5 * col[k] * lam

    # -- build trace panel --
    window_frames = min(int(WINDOW_S * fs), dff_clean.shape[1])
    traces = dff_clean[top, :window_frames]
    time = np.arange(window_frames) / fs

    # offset calculation (same logic as plot_traces)
    p10 = np.percentile(traces, 10, axis=1)
    p90 = np.percentile(traces, 90, axis=1)
    offset = max(np.median(p90 - p10) * 1.2, np.percentile(p90 - p10, 75) * 0.8, 1e-6)

    shifted = np.zeros_like(traces)
    for i in range(n_show):
        bl = np.percentile(traces[i], 8)
        shifted[i] = (traces[i] - bl) + i * offset

    # -- composite figure --
    fig = plt.figure(figsize=(20, max(8, n_show * 0.12)), facecolor="black")
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 2.2], wspace=0.05)

    # left: masks
    ax_mask = fig.add_subplot(gs[0, 0], facecolor="black")
    ax_mask.imshow(canvas, interpolation="nearest")
    ax_mask.set_title(f"Plane {pidx} â€” {n_show} ROIs", color="white",
                      fontsize=10, fontweight="bold")
    ax_mask.axis("off")

    # right: traces
    ax_tr = fig.add_subplot(gs[0, 1], facecolor="black")

    # plot from top to bottom with mask overlap
    for i in range(n_show - 1, -1, -1):
        z = n_show - i
        ax_tr.fill_between(
            time, shifted[i], y2=shifted[i].min() - offset,
            color="black", zorder=z - 0.5,
        )
        ax_tr.plot(time, shifted[i], color=colors[i], lw=0.5, zorder=z)

    ax_tr.set_xlim(0, time[-1])
    y_min, y_max = shifted.min(), shifted.max()
    y_range = y_max - y_min
    ax_tr.set_ylim(y_min - y_range * 0.02, y_max + y_range * 0.02)
    ax_tr.tick_params(axis="both", labelbottom=False, labelleft=False, length=0)
    for spine in ax_tr.spines.values():
        spine.set_visible(False)

    # time scale bar
    bar_s = round(WINDOW_S * 0.1)
    if bar_s < 1:
        bar_s = 1
    x1 = time[-1]
    x0 = x1 - bar_s
    sb_y = y_min - y_range * 0.03
    ax_tr.plot([x0, x1], [sb_y, sb_y], color="white", lw=2.5,
               solid_capstyle="butt", clip_on=False)
    ax_tr.text((x0 + x1) / 2, sb_y - y_range * 0.02, f"{bar_s} s",
               color="white", fontsize=7, ha="center", va="top")

    # vertical scale bar (10% of y range)
    vsb_h = y_range * 0.10
    vsb_x = x1 + (x1 - time[0]) * 0.01
    ax_tr.plot([vsb_x, vsb_x], [y_min, y_min + vsb_h], color="white", lw=2.5,
               solid_capstyle="butt", clip_on=False)
    ax_tr.text(vsb_x + (x1 - time[0]) * 0.015, y_min, "z-score",
               color="white", fontsize=6, va="bottom", rotation=90)

    ax_tr.set_ylabel(f"{n_show} neurons", color="white", fontsize=9,
                     fontweight="bold", labelpad=5)

    out = OUTDIR / f"plane{pidx:02d}_masks_traces.png"
    fig.savefig(out, dpi=DPI, facecolor="black", bbox_inches="tight", pad_inches=0.05)
    print(f"saved: {out}")
    plt.show()
    plt.close(fig)

print("done")

## all-planes mask overlay (single composite image)

In [ ]:
# build a single tall image: all planes stacked vertically, masks overlaid
panels = []

for pidx in range(1, NUM_PLANES + 1):
    p = load_plane(pidx)
    ops, stat, iscell = p["ops"], p["stat"], p["iscell"]
    bg = ops["meanImg"]
    Ly, Lx = ops["Ly"], ops["Lx"]

    mask_idx = iscell[:, 0].astype(bool)
    accepted_stat = [s for s, m in zip(stat, mask_idx) if m]

    # build canvas the same way plot_masks does
    vmin, vmax = np.nanpercentile(bg, [1, 99])
    norm = np.clip((bg - vmin) / (vmax - vmin + 1e-6), 0, 1)
    norm = np.nan_to_num(norm, nan=0.0)
    canvas = np.tile(norm, (3, 1, 1)).transpose(1, 2, 0).copy()

    n_masks = len(accepted_stat)
    colors = plt.cm.hsv(np.linspace(0, 1, n_masks + 1))[:, :3]
    for c_idx, s in enumerate(accepted_stat):
        yy, xx, lam = s["ypix"], s["xpix"], s["lam"]
        valid = (yy >= 0) & (yy < Ly) & (xx >= 0) & (xx < Lx)
        yy, xx, lam = yy[valid], xx[valid], lam[valid]
        lam = lam / (lam.max() + 1e-10)
        col = colors[c_idx]
        for k in range(3):
            canvas[yy, xx, k] = 0.5 * canvas[yy, xx, k] + 0.5 * col[k] * lam

    panels.append(canvas)

# arrange in 2 rows x 7 cols
ncols = 7
nrows = 2
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 4.5), facecolor="black")
for i, ax in enumerate(axes.flat):
    ax.set_facecolor("black")
    if i < len(panels):
        ax.imshow(panels[i], interpolation="nearest")
        n_acc = int(load_plane(i + 1)["iscell"][:, 0].sum())
        ax.set_title(f"P{i+1} ({n_acc})", color="white", fontsize=9, fontweight="bold")
    ax.axis("off")
    for spine in ax.spines.values():
        spine.set_visible(False)

plt.subplots_adjust(wspace=0.02, hspace=0.05)
save_img(fig, "all_planes_masks_poster")